# 05 — Model Interpretation & Error Analysis

This notebook explains **why** the model makes its predictions and examines
where it goes wrong.

## Contents
1. Environment setup & model loading
2. SHAP global feature importance (summary + bar)
3. SHAP dependence plots (top features)
4. Patient-level explanations (waterfall plots)
5. Error group classification (TP / TN / FP / FN)
6. Error summary statistics
7. Distribution plots by error type
8. Age vs Comorbidity scatter by error group
9. False positive vs false negative profiles
10. Diagnosis distribution in FP and FN
11. Interpretation & takeaways

**Model used:** best calibrated model from `models/best_tuned_model.pkl`  
**Threshold:** optimal threshold from tuning phase

## 0. Environment Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root : {ROOT}")
print(f"Python       : {sys.executable}")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from src.utils import load_config, set_seed
from src.modeling import load_features, make_splits, evaluate
from src.interpretation import (
    compute_shap_values,
    plot_shap_summary,
    plot_shap_bar,
    plot_shap_dependence,
    plot_shap_waterfall,
    compute_error_groups,
    summarise_errors,
    plot_error_distributions,
    plot_error_age_comorbidity,
    plot_false_positive_vs_negative,
    plot_error_diagnosis_distribution,
)

config = load_config(ROOT / "config" / "config.yaml")
config["_base_dir"] = str(ROOT)
# Plots saved to correct sub-directories by interpretation.py directly

SEED = config.get("random_seed", 42)
set_seed(SEED)
print("Config loaded. Random seed:", SEED)

## 1. Load Model & Features

In [ ]:
# Load best tuned model artifact
model_path = ROOT / config["paths"]["model_dir"] / "best_tuned_model.pkl"
artifact = joblib.load(model_path)

model             = artifact["model"]
feature_names     = artifact["feature_names"]
optimal_threshold = artifact["threshold"]
best_model_name   = artifact["name"]

print(f"Model        : {best_model_name}")
print(f"Threshold    : {optimal_threshold:.3f}")
print(f"Features     : {len(feature_names)}")
print(f"Val metrics  : {artifact['val_metrics']}")

In [ ]:
# Load feature matrix and splits
X, y, _ = load_features(config, base_dir=ROOT)
X_train, X_val, X_test, y_train, y_val, y_test = make_splits(X, y, config)

print(f"Val  : {len(y_val):,} rows")
print(f"Test : {len(y_test):,} rows")

In [ ]:
# Load the original (pre-OHE) cleaned dataset for error analysis
# This has the original categorical columns, age, comorbidity, etc.
from src.data_preparation import load_raw_data, clean_data, encode_features
from src.feature_engineering import create_features

raw_path = ROOT / config["paths"]["raw_data"]
df_raw   = load_raw_data(raw_path)
df_clean = clean_data(df_raw, config)
df_feat  = create_features(df_clean, config)

# Keep only rows corresponding to the validation set index
df_feat_aligned = df_feat.iloc[X_val.index].copy() if hasattr(X_val.index, '__len__') else df_feat

print(f"Pre-OHE feature DataFrame: {df_feat.shape}")

## 2. SHAP — Global Feature Importance

SHAP (SHapley Additive exPlanations) assigns each feature a contribution
score for each prediction. Positive values push the prediction toward
readmission; negative values push away from it.

> **Note:** SHAP values are computed on up to 500 validation samples for
> speed.  All relative importances are still representative.

In [ ]:
shap_values, X_shap_sample = compute_shap_values(
    model, X_val, config, max_samples=500
)
print(f"SHAP matrix shape: {shap_values.values.shape}")
print(f"Features: {shap_values.values.shape[1]}")

In [ ]:
# Beeswarm summary — shows direction AND magnitude for each feature
plot_shap_summary(shap_values, X_shap_sample, config, max_features=20)
print("Saved: reports/figures/shap/shap_summary.png")

In [ ]:
# Bar chart — mean absolute SHAP, easier to rank features
plot_shap_bar(shap_values, config, max_features=20)
print("Saved: reports/figures/shap/shap_bar_importance.png")

In [ ]:
# Top-5 features by mean |SHAP|
mean_abs   = np.abs(shap_values.values).mean(axis=0)
feat_names_shap = list(X_shap_sample.columns)
top_indices = np.argsort(mean_abs)[::-1][:10]
top_df = pd.DataFrame({
    "feature":        [feat_names_shap[i] for i in top_indices],
    "mean_|SHAP|":    mean_abs[top_indices].round(5),
})
print("Top 10 features by mean |SHAP|:")
display(top_df)

## 3. SHAP Dependence Plots — Top Features

Each dependence plot shows how a single feature's value relates to its SHAP
contribution.  The colour encodes the value of the most interacting feature
(auto-selected by SHAP).

In [ ]:
plot_shap_dependence(shap_values, X_shap_sample, config, top_n=4)
print("Dependence plots saved to reports/figures/shap/")

## 4. Patient-Level Explanations (Waterfall Plots)

Waterfall plots show how each feature shifts an individual patient's
predicted readmission probability away from the base rate.

In [ ]:
plot_shap_waterfall(shap_values, config, n_examples=3)
print("Waterfall plots saved to reports/figures/shap/")

## 5. Error Group Classification

Classify every validation-set prediction into:
- **TP** — correctly predicted readmission
- **TN** — correctly predicted no readmission
- **FP** — predicted readmission, actually not (false alarm)
- **FN** — predicted no readmission, actually readmitted (missed case)

The **optimal threshold** from tuning is used.

In [ ]:
groups = compute_error_groups(model, X_val, y_val, threshold=optimal_threshold)

total = len(y_val)
print(f"Threshold used: {optimal_threshold:.3f}")
print(f"{'Group':<6} {'N':>6} {'%':>7}")
print("-" * 22)
for name, idx in groups.items():
    print(f"{name:<6} {len(idx):>6,} {100*len(idx)/total:>6.1f}%")

## 6. Error Summary Statistics

In [ ]:
# Align original pre-OHE DataFrame to the validation set index
df_val_original = df_feat.loc[X_val.index].copy()

error_summary = summarise_errors(groups, df_val_original, config)
display(error_summary.round(2))

In [ ]:
# Highlight differences between FP and FN
if "FP" in error_summary.index and "FN" in error_summary.index:
    diff = (error_summary.loc["FP"] - error_summary.loc["FN"]).drop("n", errors="ignore")
    print("FP − FN mean difference (positive = FP higher):")
    display(diff.round(3).to_frame(name="FP − FN"))

## 7. Distribution Plots by Error Type

Violin plots compare the distribution of key clinical features across
the four prediction groups.

In [ ]:
plot_error_distributions(groups, df_val_original, config)
print("Distribution plots saved to reports/figures/error_analysis/")

## 8. Age vs Comorbidity — Error Group Scatter

In [ ]:
plot_error_age_comorbidity(groups, df_val_original, config)
print("Saved: reports/figures/error_analysis/error_age_comorbidity_scatter.png")

## 9. False Positive vs False Negative Profiles

This plot compares the **clinical profile** of patients the model
misclassifies in each direction:
- **FP**: over-predicted risk (unnecessary alert)
- **FN**: under-predicted risk (missed high-risk patient)

In [ ]:
plot_false_positive_vs_negative(groups, df_val_original, config)
print("Saved: reports/figures/error_analysis/false_positive_vs_negative.png")

## 10. Diagnosis Distribution in Errors

In [ ]:
plot_error_diagnosis_distribution(groups, df_val_original, config)
print("Saved: reports/figures/error_analysis/error_diagnosis_distribution.png")

## 11. Interpretation & Takeaways

### SHAP Findings

| Observation | Implication |
|---|---|
| Near-uniform SHAP magnitudes across all features | Consistent with synthetic dataset — no single feature dominates |
| SHAP values are small in absolute terms | Model confidence is low — predicted probabilities cluster near the base rate |
| Interaction features (Age×Comorbidity, Severity×LOS) have measurable SHAP contributions | Interaction terms add marginal signal; monitor with real data |

### Error Analysis Findings

| Group | Pattern |
|---|---|
| **FP** (false alarms) | Patients incorrectly flagged as high-risk; check if comorbidity or severity is marginally elevated |
| **FN** (missed cases) | True readmissions the model missed; typically the most clinically costly error type |
| Threshold choice | The optimal threshold (tuned for F1) pushes recall high — appropriate for a clinical safety context |

### Dataset Limitation Reminder

The dataset is **synthetic**, meaning features were generated independently
without real clinical relationships.  As a result:
- ROC-AUC hovers near 0.55 (near-chance)
- SHAP magnitudes are small and nearly equal across all features
- Error group patterns are essentially random

**All code, patterns, and architecture are production-ready.** The same pipeline
applied to real EHR data would yield meaningful SHAP rankings and clinically
interpretable error profiles.